# Pipeline — Construção da Base de Dados
## TCC: Pair Trading com S&P 500

---

| Parte | O que faz |
|---|---|
| 1 | Consolida a base do professor (1990/S2–2015/S2) com datas reais via Yahoo |
| 2 | Confirma o tipo de preço usado (Close sem dividendos) |
| 3 | Carrega composição do S&P 500 e lista tickers necessários (2016–2025) |
| 4 | Verifica quais dados já existem da pipeline anterior |
| 5 | Baixa apenas os dados que faltam (checkpoint) |
| 6 | Constrói a base de extensão (2016/S1–2025/S2) no formato do professor |
| 7 | Consolida professor + extensão na base completa final (1990/S2–2025/S2) |

In [52]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import time
from pathlib import Path

PROFESSOR_DIR = Path("../data_bases/professor")
OUTPUT_DIR    = Path("../data_bases")
EXTERNAL_DIR  = Path("../data_bases/external")
PRICES_YAHOO  = Path("../data_bases/prices")
PRICES_TIINGO = Path("../data_bases/prices_tiingo")
OUTPUT_DIR.mkdir(exist_ok=True)

TIINGO_TOKEN = "dadfd331f2cb44969b8f7468006d20ad62b13262"

print("Pronto.")

Pronto.


---
## Parte 1 — Consolidar a base do professor

A base original (`Pt.csv`) não tem datas. Usamos o histórico do MSFT no Yahoo como calendário NYSE real e encontramos por correlação onde os preços do professor terminam. Sem `bdate_range`, sem deriva de feriados.

In [53]:
periods = pd.read_csv(PROFESSOR_DIR / "Periods.csv", header=None)
periods.columns = ["dias_sem", "col2", "col3", "dias_ano"]
periods = periods[["dias_sem"]].copy()

pt = pd.read_csv(PROFESSOR_DIR / "Pt.csv")

n_rows = min(int(periods["dias_sem"].sum()), len(pt))  # 6425

print(f"Periods.csv: {len(periods)} semestres, {periods['dias_sem'].sum()} dias")
print(f"Pt.csv:      {len(pt)} linhas × {pt.shape[1]} tickers")
print(f"Linhas a usar: {n_rows}")

Periods.csv: 51 semestres, 6425 dias
Pt.csv:      6426 linhas × 1100 tickers
Linhas a usar: 6425


In [54]:
# Baixar histórico completo do MSFT — as datas dele são o calendário real da NYSE
raw_full = yf.download("MSFT", start="1990-07-01", end="2016-01-05",
                       auto_adjust=False, progress=False)

if isinstance(raw_full.columns, pd.MultiIndex):
    msft_yahoo = raw_full[("Close", "MSFT")]
else:
    msft_yahoo = raw_full["Close"]

msft_yahoo.index = pd.to_datetime(msft_yahoo.index).tz_localize(None)

print(f"Calendário NYSE: {msft_yahoo.index[0].date()} → {msft_yahoo.index[-1].date()} ({len(msft_yahoo)} dias)")

Calendário NYSE: 1990-07-02 → 2016-01-04 (6428 dias)


In [55]:
# Correlação deslizante: encontra onde os últimos 40 valores do professor
# se encaixam no histórico do Yahoo
N = 40
prof_msft    = pt["MSFT"].iloc[:n_rows].tail(N).values
search_start = len(msft_yahoo) - 300

best_end_pos = search_start
best_corr    = -1.0

for end_pos in range(search_start, len(msft_yahoo)):
    yahoo_window = msft_yahoo.iloc[end_pos - N + 1 : end_pos + 1].values
    corr = np.corrcoef(prof_msft, yahoo_window)[0, 1]
    if corr > best_corr:
        best_corr   = corr
        best_end_pos = end_pos

true_last_date = msft_yahoo.index[best_end_pos]

print(f"Correlação:      {best_corr:.6f}")
print(f"Último dia real: {true_last_date.date()}")
print()

matched = msft_yahoo.iloc[best_end_pos - N + 1 : best_end_pos + 1].tail(5)
check = pd.DataFrame({
    "professor": pt["MSFT"].iloc[:n_rows].tail(5).values,
    "yahoo":     matched.values,
}, index=matched.index)
check["ratio"] = check["professor"] / check["yahoo"]
print("Conferência (ratio deve ser = 1.0):")
print(check)

Correlação:      0.990998
Último dia real: 2015-12-30

Conferência (ratio deve ser = 1.0):
            professor      yahoo  ratio
Date                                   
2015-12-23      55.82  55.820000    1.0
2015-12-24      55.67  55.669998    1.0
2015-12-28      55.95  55.950001    1.0
2015-12-29      56.55  56.549999    1.0
2015-12-30      56.31  56.310001    1.0


In [56]:
# Datas reais de cada linha da base do professor
true_dates = msft_yahoo.index[best_end_pos - n_rows + 1 : best_end_pos + 1]

# Metadados de semestre
periodo_info = []
year, sem = 1990, 2
for i in range(len(periods)):
    n_dias = int(periods.iloc[i]["dias_sem"])
    for dia_idx in range(n_dias):
        periodo_info.append({
            "year": year, "semester": sem, "period": f"{year}/S{sem}",
            "day_in_semester": dia_idx + 1, "total_days_sem": n_dias,
        })
    if sem == 2:
        year += 1; sem = 1
    else:
        sem = 2

date_map = pd.DataFrame(periodo_info[:len(true_dates)])
date_map["date"] = true_dates.values

dias_2001s2 = (date_map["period"] == "2001/S2").sum()
print(f"Datas: {true_dates[0].date()} → {true_dates[-1].date()} ({len(true_dates)} dias)")
print(f"2001/S2 → {dias_2001s2} dias (esperado: 123) {'✓' if dias_2001s2 == 123 else '✗'}")

Datas: 1990-07-03 → 2015-12-30 (6425 dias)
2001/S2 → 123 dias (esperado: 123) ✓


In [57]:
# Montar e salvar a base consolidada
pt_consolidado = pd.concat([
    date_map[["date", "year", "semester", "period", "day_in_semester", "total_days_sem"]].reset_index(drop=True),
    pt.iloc[:n_rows].reset_index(drop=True)
], axis=1).set_index("date")
pt_consolidado.index = pd.to_datetime(pt_consolidado.index)

print(f"Base consolidada: {pt_consolidado.shape}")
print(pt_consolidado[["year", "semester", "period", "MSFT", "KO", "JNJ"]].tail(3))

pt_consolidado.to_csv(OUTPUT_DIR / "professor_consolidado.csv")
print(f"\nSalvo: professor_consolidado.csv")

Base consolidada: (6425, 1105)
            year  semester   period   MSFT     KO     JNJ
date                                                     
2015-12-28  2015         2  2015/S2  55.95  43.49  103.22
2015-12-29  2015         2  2015/S2  56.55  43.71  104.03
2015-12-30  2015         2  2015/S2  56.31  43.57  103.78

Salvo: professor_consolidado.csv


---
## Parte 2 — Confirmar tipo de preço (Close sem dividendos)

Com datas exatas, o ratio `professor / yahoo_close` deve ser **1.00000** e `std = 0`.

In [58]:
TICKERS   = ["MSFT", "KO", "JNJ"]
prof_ult  = pt_consolidado[TICKERS].tail(20).copy()
start_str = prof_ult.index[0].strftime("%Y-%m-%d")
end_str   = prof_ult.index[-1].strftime("%Y-%m-%d")

# Yahoo
yc, ya = {}, {}
for t in TICKERS:
    raw = yf.download(t, start=start_str, end=end_str, auto_adjust=False, progress=False)
    close     = raw[("Close",     t)] if isinstance(raw.columns, pd.MultiIndex) else raw["Close"]
    adj_close = raw[("Adj Close", t)] if isinstance(raw.columns, pd.MultiIndex) else raw["Adj Close"]
    close.index     = pd.to_datetime(close.index).tz_localize(None)
    adj_close.index = pd.to_datetime(adj_close.index).tz_localize(None)
    yc[t], ya[t]    = close, adj_close
yc, ya = pd.DataFrame(yc), pd.DataFrame(ya)

# Tiingo
def fetch_tiingo(ticker, start, end, token):
    r = requests.get(
        f"https://api.tiingo.com/tiingo/daily/{ticker}/prices",
        params={"startDate": start, "endDate": end, "token": token, "resampleFreq": "daily"},
        timeout=15
    )
    if r.status_code != 200:
        return None
    df = pd.DataFrame(r.json())
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    return df.set_index("date")[["close", "adjClose"]]

tc, ta = {}, {}
for t in TICKERS:
    res = fetch_tiingo(t, start_str, end_str, TIINGO_TOKEN)
    if res is not None:
        tc[t] = res["close"].rename(t)
        ta[t] = res["adjClose"].rename(t)
tc, ta = pd.DataFrame(tc), pd.DataFrame(ta)

# Comparação
print("=" * 60)
for t in TICKERS:
    cmp = pd.DataFrame({"p": prof_ult[t], "yc": yc.get(t), "ya": ya.get(t),
                        "tc": tc.get(t), "ta": ta.get(t)}).dropna()
    print(f"{t}: ratio/yahoo_close={( cmp.p/cmp.yc).mean():.5f} std={(cmp.p/cmp.yc).std():.5f} | "
          f"ratio/tiingo_close={(cmp.p/cmp.tc).mean():.5f} std={(cmp.p/cmp.tc).std():.5f} "
          f"({len(cmp)} dias)")
print("\nConclusão: usar Close (auto_adjust=False) — sem desconto de dividendos.")

MSFT: ratio/yahoo_close=1.00000 std=0.00000 | ratio/tiingo_close=1.00000 std=0.00000 (19 dias)
KO: ratio/yahoo_close=1.00000 std=0.00000 | ratio/tiingo_close=1.00000 std=0.00000 (19 dias)
JNJ: ratio/yahoo_close=1.00000 std=0.00000 | ratio/tiingo_close=1.00000 std=0.00000 (19 dias)

Conclusão: usar Close (auto_adjust=False) — sem desconto de dividendos.


---
## Parte 3 — Composição do S&P 500 e tickers necessários (2016–2025)

Carregamos a composição histórica do S&P 500 e identificamos todos os tickers que estiveram no índice durante os semestres de 2016/S1 a 2025/S2.

In [59]:
hist_sp500 = pd.read_csv(EXTERNAL_DIR / "sp500_historico.csv")
hist_sp500["date"] = pd.to_datetime(hist_sp500["date"])

print(f"Composição: {len(hist_sp500)} snapshots")
print(f"Período:    {hist_sp500['date'].min().date()} → {hist_sp500['date'].max().date()}")

def get_constituents(date_str):
    """Retorna os tickers do S&P 500 na data indicada."""
    date   = pd.to_datetime(date_str)
    subset = hist_sp500[hist_sp500["date"] <= date]
    if len(subset) == 0:
        return set()
    return set(subset.iloc[-1]["tickers"].split(","))

# Semestres da extensão: 2016/S1 a 2025/S2
# O professor termina em 2015-12-30, então a extensão começa em 2016
semesters_ext = []
for year in range(2016, 2026):
    for sem in [1, 2]:
        inicio = f"{year}-01-01" if sem == 1 else f"{year}-07-01"
        fim    = f"{year}-06-30" if sem == 1 else f"{year}-12-31"
        semesters_ext.append({
            "period": f"{year}/S{sem}", "year": year, "semester": sem,
            "inicio": inicio, "fim": fim,
        })

# Todos os tickers únicos necessários
todos_tickers = sorted({t for s in semesters_ext for t in get_constituents(s["fim"])})

print(f"\nSemestres da extensão: {len(semesters_ext)} (2016/S1 → 2025/S2)")
print(f"Tickers únicos necessários: {len(todos_tickers)}")
print(f"Exemplos: {todos_tickers[:10]}")

Composição: 2705 snapshots
Período:    1996-01-02 → 2026-01-14

Semestres da extensão: 20 (2016/S1 → 2025/S2)
Tickers únicos necessários: 708
Exemplos: ['A', 'AABA', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABMD', 'ABNB', 'ABT']


---
## Parte 4 — Verificar cobertura dos dados já baixados

A pipeline anterior já baixou dados de 2015-07-01 a 2025-12-31 para os tickers do S&P 500. Os arquivos têm formato simples: `Date, {ticker}` (só `Close`, sem dividendos). Verificamos quantos dos tickers necessários já têm arquivo.

In [76]:
# Para cada ticker, verificar onde os dados existem
cobertura = {}
for ticker in todos_tickers:
    if   (PRICES_YAHOO  / f"{ticker}.csv").exists():
        cobertura[ticker] = "yahoo"
    elif (PRICES_TIINGO / f"{ticker}.csv").exists():
        cobertura[ticker] = "tiingo"
    else:
        cobertura[ticker] = "falta"

n_yahoo  = sum(1 for v in cobertura.values() if v == "yahoo")
n_tiingo = sum(1 for v in cobertura.values() if v == "tiingo")
n_falta  = sum(1 for v in cobertura.values() if v == "falta")

print(f"Tickers necessários: {len(todos_tickers)}")
print(f"  Yahoo já baixado:   {n_yahoo}")
print(f"  Tiingo já baixado:  {n_tiingo}")
print(f"  Sem dados ainda:    {n_falta}")

if n_falta > 0:
    faltam = [t for t, v in cobertura.items() if v == "falta"]
    print(f"\nTickers sem dados: {faltam}")

Tickers necessários: 708
  Yahoo já baixado:   597
  Tiingo já baixado:  80
  Sem dados ainda:    31

Tickers sem dados: ['ANTM', 'APC', 'ARNC', 'BLL', 'CA', 'CBS', 'CDAY', 'DISCA', 'DNB', 'DO', 'EMC', 'ENDP', 'FB', 'FBHS', 'FRC', 'FTR', 'GPS', 'HCP', 'LB', 'MMC', 'MNK', 'MON', 'PEAK', 'PKI', 'RE', 'SE', 'STI', 'TE', 'TMK', 'VIAC', 'WRK']


In [77]:
# Conferir formato dos arquivos existentes
print("Formato Yahoo (MSFT.csv):")
df_yahoo = pd.read_csv(PRICES_YAHOO / "MSFT.csv")
df_yahoo["Date"] = pd.to_datetime(df_yahoo["Date"])
print(f"  {len(df_yahoo)} linhas | {df_yahoo['Date'].iloc[0].date()} → {df_yahoo['Date'].iloc[-1].date()}")
print(f"  Colunas: {df_yahoo.columns.tolist()}")
print(df_yahoo.head(3).to_string())

# Tiingo (se existir)
tiingo_files = list(PRICES_TIINGO.glob("*.csv"))
if tiingo_files:
    print(f"\nFormato Tiingo ({tiingo_files[0].name}):")
    df_tiingo = pd.read_csv(tiingo_files[0])
    df_tiingo.columns = [c.lower() for c in df_tiingo.columns]
    date_col = "date" if "date" in df_tiingo.columns else df_tiingo.columns[0]
    df_tiingo[date_col] = pd.to_datetime(df_tiingo[date_col])
    print(f"  {len(df_tiingo)} linhas | {df_tiingo[date_col].iloc[0].date()} → {df_tiingo[date_col].iloc[-1].date()}")
    print(f"  Colunas: {df_tiingo.columns.tolist()}")
    print(df_tiingo.head(3).to_string())

Formato Yahoo (MSFT.csv):
  2641 linhas | 2015-07-01 → 2025-12-30
  Colunas: ['Date', 'MSFT']
        Date       MSFT
0 2015-07-01  44.450001
1 2015-07-02  44.400002
2 2015-07-06  44.389999

Formato Tiingo (AABA.csv):
  380 linhas | 2015-07-01 → 2016-12-30
  Colunas: ['date', 'aaba']
        date   aaba
0 2015-07-01  39.33
1 2015-07-02  39.38
2 2015-07-06  38.61


---
## Parte 5 — Baixar dados faltantes (checkpoint)

Para os tickers sem arquivo, tentamos baixar do Yahoo Finance. Se o dado não existir no Yahoo (empresa adquirida, falida, ticker renomeado), registramos como `delisted`.

**Checkpoint:** se a célula for interrompida, ao rodar novamente ela pula os tickers já tentados.

In [78]:
CHECKPOINT = EXTERNAL_DIR / "coleta_report.csv"

# Carregar checkpoint anterior (se existir)
if CHECKPOINT.exists():
    relatorio   = pd.read_csv(CHECKPOINT)
    ja_tentados = set(relatorio["ticker"])
    print(f"Checkpoint carregado: {len(ja_tentados)} tickers já tentados.")
else:
    relatorio   = pd.DataFrame(columns=["ticker", "status", "n_rows", "note"])
    ja_tentados = set()

# Tickers que faltam E ainda não foram tentados
para_baixar = [t for t, v in cobertura.items() if v == "falta" and t not in ja_tentados]
print(f"Para baixar agora: {len(para_baixar)}")

for ticker in para_baixar:
    print(f"  {ticker}... ", end="", flush=True)
    try:
        raw = yf.download(ticker, start="2015-07-01", end="2026-01-01",
                          auto_adjust=False, progress=False)

        # Extrair apenas o Close (sem dividendos)
        if isinstance(raw.columns, pd.MultiIndex):
            close = raw[("Close", ticker)].rename(ticker)
        else:
            close = raw["Close"].rename(ticker)

        if len(close) > 10:
            close.to_csv(PRICES_YAHOO / f"{ticker}.csv", header=True)
            row = {"ticker": ticker, "status": "ok", "n_rows": len(close), "note": ""}
            print(f"ok ({len(close)} linhas)")
            cobertura[ticker] = "yahoo"  # atualiza cobertura
        else:
            row = {"ticker": ticker, "status": "delisted", "n_rows": len(close), "note": "sem dados"}
            print("delisted")

    except Exception as e:
        row = {"ticker": ticker, "status": "error", "n_rows": 0, "note": str(e)[:100]}
        print(f"erro")

    relatorio   = pd.concat([relatorio, pd.DataFrame([row])], ignore_index=True)
    relatorio.to_csv(CHECKPOINT, index=False)
    ja_tentados.add(ticker)
    time.sleep(0.3)

print()
# Resumo final
n_falta_final = sum(1 for v in cobertura.values() if v == "falta")
print("=== Resumo final ===")
print(f"Yahoo:   {sum(1 for v in cobertura.values() if v == 'yahoo')}")
print(f"Tiingo:  {sum(1 for v in cobertura.values() if v == 'tiingo')}")
print(f"Faltam:  {n_falta_final}  ← empresas não encontradas em nenhuma fonte")
if relatorio["status"].isin(["ok","delisted","error"]).any():
    print()
    print(relatorio["status"].value_counts().to_string())

Checkpoint carregado: 30 tickers já tentados.
Para baixar agora: 1
  VIAC... 

$VIAC: possibly delisted; no timezone found

1 Failed download:
['VIAC']: possibly delisted; no timezone found


delisted

=== Resumo final ===
Yahoo:   597
Tiingo:  80
Faltam:  31  ← empresas não encontradas em nenhuma fonte

status
delisted    24
ok           7


---
## Parte 6 — Construir a base de extensão (2016/S1–2025/S2)

Formato idêntico ao professor: datas como índice, metadados de semestre, e um ticker por coluna.  
Cada ticker só tem valor nas datas em que efetivamente integrava o S&P 500.

In [79]:
# Calendário NYSE real da extensão: MSFT já foi baixado com datas precisas
msft_hist = pd.read_csv(PRICES_YAHOO / "MSFT.csv", index_col=0, parse_dates=True)
msft_hist.index = pd.to_datetime(msft_hist.index, utc=True).tz_convert(None).normalize()
trading_dates_ext = msft_hist.index[msft_hist.index >= "2016-01-01"].sort_values()

print(f"Calendário real 2016–2025: {len(trading_dates_ext)} dias úteis")
print(f"  {trading_dates_ext[0].date()} → {trading_dates_ext[-1].date()}")

# Carregar preços dos arquivos existentes (um CSV por ticker)
print("\nCarregando preços... ", end="", flush=True)
all_prices = {}
falhas = []
for ticker, source in cobertura.items():
    if source not in ("yahoo", "tiingo"):
        continue
    try:
        if source == "yahoo":
            df = pd.read_csv(PRICES_YAHOO  / f"{ticker}.csv", index_col=0, parse_dates=True)
        else:
            df = pd.read_csv(PRICES_TIINGO / f"{ticker}.csv", index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index, utc=True).tz_convert(None).normalize()
        serie = df.iloc[:, 0]  # coluna de preço (única)
        all_prices[ticker] = serie[serie.index >= "2016-01-01"]
    except Exception as e:
        falhas.append((ticker, str(e)[:60]))
print(f"{len(all_prices)} tickers carregados")
if falhas:
    print(f"Falhas de leitura: {falhas}")

# Metadados de semestre para cada dia de negociação (igual à Parte 1, mas para 2016–2025)
meta_rows = []
for s in semesters_ext:
    mask      = (trading_dates_ext >= s["inicio"]) & (trading_dates_ext <= s["fim"])
    datas_sem = trading_dates_ext[mask]
    n         = len(datas_sem)
    for j, d in enumerate(datas_sem):
        meta_rows.append({
            "date": d, "period": s["period"], "year": s["year"],
            "semester": s["semester"], "day_in_semester": j + 1, "total_days_sem": n,
        })

meta = pd.DataFrame(meta_rows).set_index("date")
meta.index = pd.to_datetime(meta.index)

# Máscara de pertencimento: True onde o ticker estava no S&P 500 naquele semestre
membership = pd.DataFrame(False, index=meta.index, columns=todos_tickers)
for s in semesters_ext:
    tickers_sem = [t for t in get_constituents(s["fim"]) if t in membership.columns]
    date_mask   = (meta.index >= s["inicio"]) & (meta.index <= s["fim"])
    membership.loc[date_mask, tickers_sem] = True

# Montar base wide: preços alinhados ao calendário, mascarados pela composição
prices_wide = pd.DataFrame(all_prices).reindex(index=meta.index, columns=todos_tickers)
extensao    = pd.concat([meta, prices_wide.where(membership)], axis=1)

print(f"\nBase de extensão: {extensao.shape}")
print(extensao[["period", "year", "semester", "MSFT", "AAPL", "JPM"]].head(3).to_string())
print("...")
print(extensao[["period", "year", "semester", "MSFT", "AAPL", "JPM"]].tail(3).to_string())

extensao.to_csv(OUTPUT_DIR / "extensao_2016_2025.csv")
print(f"\nSalvo: extensao_2016_2025.csv")

Calendário real 2016–2025: 2513 dias úteis
  2016-01-04 → 2025-12-30

Carregando preços... 677 tickers carregados

Base de extensão: (2513, 713)
             period  year  semester       MSFT       AAPL        JPM
date                                                                
2016-01-04  2016/S1  2016         1  54.799999  26.337500  63.619999
2016-01-05  2016/S1  2016         1  55.049999  25.677500  63.730000
2016-01-06  2016/S1  2016         1  54.049999  25.174999  62.810001
...
             period  year  semester        MSFT        AAPL         JPM
date                                                                   
2025-12-26  2025/S2  2025         2  487.709991  273.399994  327.910004
2025-12-29  2025/S2  2025         2  487.100006  273.760010  323.750000
2025-12-30  2025/S2  2025         2  487.480011  273.079987  323.420013

Salvo: extensao_2016_2025.csv


---
## Parte 7 — Consolidar base completa (1990/S2–2025/S2)

Concatena a base do professor com a extensão. O pandas alinha as colunas automaticamente: tickers que só existem num período ficam com NaN no outro.

In [80]:
prof = pd.read_csv(OUTPUT_DIR / "professor_consolidado.csv", index_col=0, parse_dates=True)
ext  = pd.read_csv(OUTPUT_DIR / "extensao_2016_2025.csv",    index_col=0, parse_dates=True)

# Concatenar: pandas alinha colunas automaticamente, NaN onde não há dados
base_completa = pd.concat([prof, ext]).sort_index()

META_COLS   = {"year", "semester", "period", "day_in_semester", "total_days_sem"}
n_tickers   = len([c for c in base_completa.columns if c not in META_COLS])

print(f"Professor    (1990–2015): {prof.shape[0]:>5} dias | {prof.shape[1] - 5:>4} tickers")
print(f"Extensão     (2016–2025): {ext.shape[0]:>5} dias | {ext.shape[1] - 5:>4} tickers")
print(f"Base completa           : {base_completa.shape[0]:>5} dias | {n_tickers:>4} tickers únicos")
print(f"  {base_completa.index[0].date()} → {base_completa.index[-1].date()}")
print(f"  {base_completa['period'].nunique()} semestres (1990/S2 → 2025/S2)")
print()

# Amostra nas junções
print(base_completa[["year", "period", "MSFT", "AAPL", "JNJ"]].iloc[[0, 1, 6423, 6424, 6425, 6426, -2, -1]].to_string())

base_completa.to_csv(OUTPUT_DIR / "base_completa.csv")
print(f"\nSalvo: base_completa.csv")

Professor    (1990–2015):  6425 dias | 1100 tickers
Extensão     (2016–2025):  2513 dias |  708 tickers
Base completa           :  8938 dias | 1343 tickers únicos
  1990-07-03 → 2025-12-30
  71 semestres (1990/S2 → 2025/S2)

            year   period        MSFT        AAPL         JNJ
date                                                         
1990-07-03  1990  1990/S2    0.714800    1.375300    4.937300
1990-07-05  1990  1990/S2    0.697800    1.375300    4.928400
2015-12-29  2015  2015/S2   56.550000  108.740000  104.030000
2015-12-30  2015  2015/S2   56.310000  107.320000  103.780000
2016-01-04  2016  2016/S1   54.799999   26.337500  100.480003
2016-01-05  2016  2016/S1   55.049999   25.677500  100.900002
2025-12-29  2025  2025/S2  487.100006  273.760010  207.559998
2025-12-30  2025  2025/S2  487.480011  273.079987  206.910004

Salvo: base_completa.csv
